# kaggle video server (comfyui)

headless comfyui with a named model stack, exposed through cloudflared — the url serves the **full comfyui node gui**, so you can build workflows in the browser or drive it headless via `queue_workflow`.

edit `VIDEO_KEY` in the config cell and run top to bottom. first run downloads 15-25GB — expect several minutes.

In [ ]:
# --- setup: pull in the shared modules -----------------------------------

# option a: clone from the repo.
# rm -rf first so re-running this cell doesn't fail on an existing dir.
!rm -rf /kaggle/working/harness_src && git clone https://github.com/Meru143/kaggle-model-server.git /kaggle/working/harness_src
import sys
sys.path.append("/kaggle/working/harness_src")

# option b: attach the .py files as a Kaggle Dataset instead --
# comment out option a above and uncomment this line instead:
# sys.path.append("/kaggle/input/TODO-your-dataset-slug")

!pip install -q huggingface_hub requests

In [ ]:
# re-running setup + this cell picks up freshly-pulled code without a
# kernel restart (python caches imports; plain re-import returns stale code)
import importlib, sys
for _m in ("comfy_bootstrap", "harness"):
    if _m in sys.modules:
        try:
            sys.modules[_m].stop()  # old modules own any running procs
        except Exception:
            pass
for _m in ("harness", "comfy_bootstrap"):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

import comfy_bootstrap as comfy

list(comfy.STACKS.keys())

## config — change this line to swap stacks

In [ ]:
VIDEO_KEY = "ltx-2.3"  # <- "ltx-2.3", "scail-2", or "lingbot-30b" (experimental)

In [ ]:
comfy.install()            # clone comfyui + gguf/kjnodes packs (idempotent)
comfy.fetch_stack(VIDEO_KEY)  # download + symlink the model set
url = comfy.start()        # boots headless, tunnels, returns the public gui url
url

In [ ]:
# headless driving (optional): in the gui use "Export (API format)", upload
# the json to kaggle, then:
# import json
# paths = comfy.queue_workflow(json.load(open("/kaggle/input/YOUR-DATASET/workflow_api.json")))
# paths  # <- output files under /kaggle/tmp/ComfyUI/output/

call `comfy.stop()` in a new cell to tear comfyui down (the tunnel is managed by the harness — `harness.stop()` clears it).

if it fails to come up, the log is `/kaggle/tmp/comfyui.log` (`!tail -50 /kaggle/tmp/comfyui.log`).

the lingbot-30b stack wants **structured json captions** (use the node pack's Structured Prompt node, always set lighting) — a ready workflow json ships in its gguf repo.